In [1]:
!pip install -q chromadb sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


In [2]:
from google.colab import files

print("Sube tus archivos PDF (manual_ti.pdf y manual_rrhh.pdf):")
subidos = files.upload()

Sube tus archivos PDF (manual_ti.pdf y manual_rrhh.pdf):


Saving manual_rrhh.pdf to manual_rrhh.pdf
Saving manual_ti.pdf to manual_ti.pdf


In [3]:
# Instalar pypdf si no está disponible
!pip install -q pypdf

import os
from pypdf import PdfReader
import chromadb
from sentence_transformers import SentenceTransformer

# Listas dinámicas para almacenar el contenido extraído
textos = []
ids = []
metadatas = []

# Recorrer los archivos PDF que acabas de subir
for filename in subidos.keys():
    print(f"Procesando archivo: {filename}")
    lector = PdfReader(filename)

    # Extraer texto página por página
    for num_pagina, pagina in enumerate(lector.pages):
        texto_pagina = pagina.extract_text()

        # Omitir páginas en blanco
        if texto_pagina and texto_pagina.strip():
            textos.append(texto_pagina)
            ids.append(f"{filename}_pag_{num_pagina + 1}")
            metadatas.append({
                "fuente": filename,
                "pagina": num_pagina + 1
            })

print(f"\nTotal de páginas extraídas y listas para indexar: {len(textos)}")

# 1) Cargar el modelo de embeddings
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# 2) Calcular vectores de todo el contenido de los PDFs
print("Calculando vectores...")
vectores_calculados = modelo.encode(textos, normalize_embeddings=True).tolist()

# 3) Configurar ChromaDB (Persistente local en Colab)
cliente = chromadb.PersistentClient(path="./chroma_db")
coleccion = cliente.get_or_create_collection(
    name="manuales",
    metadata={"hnsw:space": "cosine"},
)

# 4) Guardar en la base vectorial con upsert
coleccion.upsert(
    ids=ids,
    documents=textos,
    metadatas=metadatas,
    embeddings=vectores_calculados,
)
print("¡Indexación completada en ChromaDB!")

# 5) Prueba de consulta (asegúrate de cambiar "manual_ti.pdf" por el nombre exacto de tu archivo)
nombre_archivo_ti = "manual_ti.pdf" # Cambia esto si tu archivo tiene otro nombre exacto

vector_pregunta = modelo.encode(["¿Cómo recupero mi contraseña?"], normalize_embeddings=True).tolist()
resultado = coleccion.query(
    query_embeddings=vector_pregunta,
    n_results=2,
    where={"fuente": nombre_archivo_ti},
)

print("\n--- Resultados de la búsqueda en tu PDF ---")
for doc, meta in zip(resultado["documents"][0], resultado["metadatas"][0]):
    print(f"\n[Fuente: {meta['fuente']} - Página {meta['pagina']}]")
    print(doc[:300] + "...") # Muestra un extracto del texto encontrado

Procesando archivo: manual_rrhh.pdf
Procesando archivo: manual_ti.pdf

Total de páginas extraídas y listas para indexar: 30


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Calculando vectores...
¡Indexación completada en ChromaDB!

--- Resultados de la búsqueda en tu PDF ---

[Fuente: manual_ti.pdf - Página 11]
11. Gestión de Parches y Actualizaciones de Software
Mantener los sistemas operativos, aplicaciones y ﬁrmware actualizados es una medida primaria para mitigar
vulnerabilidades conocidas antes de que puedan ser explotadas por atacantes.
11.1 Ventanas de Parcheo por Criticidad
Severidad de Vulnerabili...

[Fuente: manual_ti.pdf - Página 5]
5. Respaldo de Datos y Continuidad del Negocio (DRP)
El Plan de Recuperación ante Desastres (DRP) establece las estrategias técnicas necesarias para restaurar los
servicios críticos de información en caso de contingencias mayores, catástrofes naturales o ataques de
Ransomware.
5.1 Objetivos de Recup...
